In [1]:
# Get raw data
with open('input/20.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
# Part 1
class Machine(object):
    def __init__(self, module_map):
        self.module_map = module_map
        self.queue = list()
        self.n_pulses = {'L':0, 'H':0}
        self.mpophist = set()
        self.modules = {'broadcaster': BCMod(self, 'broadcaster')}
        self.modules['broadcaster'].get_connections()
        del(self.mpophist)
        
    def start(self):
        self.queue = [[None, 'L', [self.modules['broadcaster']]]]
        self.mod_n_pulses = dict()
        while self.queue:
            f,p,ts = self.queue.pop(0)
            if f:
                if f.name not in self.mod_n_pulses:
                    self.mod_n_pulses[f.name] = {'L':0, 'H':0}
                self.mod_n_pulses[f.name][p] += 1
            for t in ts:
                t.process(p,f)
                self.n_pulses[p] += 1
            
class Module(object):
    def __init__(self, m, name, mtype):
        self.m = m
        self.name = name
        self.mtype = mtype
        
    def __repr__(self):
        return f'{self.name} ({self.mtype})'
    
    def get_connections(self):
        self.connect = []
        for i in self.m.module_map[self.name][1]:
            if (self.name,i) not in self.m.mpophist:
                self.m.mpophist.add((self.name,i))
                mtype = self.m.module_map[i][0]
                if i not in self.m.modules:
                    self.m.modules[i] = {'b':BCMod,'%':FFMod,'&':CJMod}[mtype](self.m, i)
                    self.m.modules[i].get_connections()
                if mtype == '&':
                    self.m.modules[i].mem_init(self)
                self.connect += [self.m.modules[i]]
    
class BCMod(Module):
    def __init__(self, m, name):
        super().__init__(m, name, 'broadcaster')
        self.lp_rec = False
        
    def process(self, pulse, _):
        if pulse=='L':
            self.lp_rec = True
        self.m.queue.extend([[self, 
                              pulse, 
                              [i for i in self.connect]]])

class FFMod(Module):
    def __init__(self, m, name):
        super().__init__(m, name, 'flipflop')
        self.state = 0

    def process(self, pulse, _):
        if pulse=='L':
            self.state = 1-self.state
            self.m.queue.extend([[self,
                                  'LH'[self.state],
                                  [i for i in self.connect]]])

class CJMod(Module):
    def __init__(self, m, name):
        super().__init__(m, name, 'conjunction')
        self.last_rec = dict()
        
    def mem_init(self, inmod):
        self.last_rec[inmod.name] = 'L'

    def process(self, pulse, inmod):
        self.last_rec[inmod.name] = pulse
        pulse_out = 'HL'[{*self.last_rec.values()} == {'H'}]
        self.m.queue.extend([[self,
                              pulse_out,
                              [i for i in self.connect]]])

module_map = {(z:=j[0])[(z[0] in '%&'):]: (z[0], j[1].split(', '))
              for i in rawinput.split('\n')
              for j in [i.split(' -> ')]}
module_map.update({j: ('b', []) 
                   for i in module_map.values() 
                   for j in i[1] 
                   if j not in module_map})

machine = Machine(module_map)
for i in range(1000):
    machine.start()
machine.n_pulses['L'] * machine.n_pulses['H']

737679780

In [3]:
# Part 2
last_conj = [k for k,v in module_map.items() if 'rx' in v[1]][0]
last_conj_inputs = [k for k,v in module_map.items() if last_conj in v[1]]

machine = Machine(module_map)
req_loops = 1
for i in range(2**12):
    machine.start()
    for j in last_conj_inputs:
        if machine.mod_n_pulses[j]['H']:
            req_loops *= (i+1)

req_loops

227411378431763